In [0]:
from pyspark.sql import functions as F

storage_account = "stretailcdcproj"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/retail_orders/"
gold_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/sales_summary/"

# 1. Read the clean Silver data
df_silver = spark.read.format("delta").load(silver_path)

# 2. Aggregate data for the Gold Layer (Reporting ready)
df_gold = (
    df_silver
    .groupBy("region", "product")
    .agg(
        F.sum("amount").alias("total_sales"),
        F.count("order_id").alias("total_orders"),
        F.round(F.avg("amount"), 2).alias("avg_order_value")
    )
    .orderBy("region", F.col("total_sales").desc())
)

# 3. Write to Gold container
df_gold.write.mode("overwrite").format("delta").save(gold_path)

print(f"Gold layer aggregated! Wrote {df_gold.count()} summary records.")
df_gold.show()

✅ Gold layer aggregated! Wrote 24 summary records.
+------+-----------+------------------+------------+---------------+
|region|    product|       total_sales|total_orders|avg_order_value|
+------+-----------+------------------+------------+---------------+
|  East|  Sunscreen| 59156.46000000001|         228|         262.92|
|  East| Face Cream|55751.859999999986|         210|         265.49|
|  East|      Serum| 54468.98000000001|         209|         263.14|
|  East|   Lipstick|48638.520000000026|         184|         271.72|
|  East|Conditioner| 48156.02999999999|         202|         243.21|
|  East|    Shampoo|48097.560000000005|         196|         247.93|
| North|   Lipstick|          61173.64|         251|         245.68|
| North| Face Cream|56384.450000000026|         229|         251.72|
| North|  Sunscreen| 55462.94000000001|         220|          252.1|
| North|    Shampoo| 53950.24999999999|         207|         260.63|
| North|      Serum|          53711.45|         210|